
# **Lab 5: Predicting Diversions (Classification with BQML)**
**Unit 2 • Week 8 (Thu) — Classification & Evaluation**

**Objective:** Train and evaluate a **logistic regression** model to classify whether a flight will be **diverted**. Interpret **precision/recall** and the **confusion matrix**, and practice threshold tuning.


## Setup & Authentication

In [3]:

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "big-data-analysis-472319"
FULL_TABLE = "bigquery-samples.airline_ontime_data.flights"  # update if needed

client = bigquery.Client(project=PROJECT_ID)
print("Authenticated. Project:", PROJECT_ID)
print("Using table:", FULL_TABLE)


Authenticated. Project: big-data-analysis-472319
Using table: bigquery-samples.airline_ontime_data.flights



---
## Business Context

> An airline wants to proactively identify flights with a high probability of being **diverted** to better manage logistics and passenger communication.

**Question:** Which is more costly for the airline: a **false positive** (predict diversion, but no diversion) or a **false negative** (fail to predict a diversion that occurs)?  
Write your reasoning below in 4–6 sentences.



---
## Train a Classification Model (LOGISTIC_REG)

Use BQML to train a **logistic regression** model predicting `diverted` using a few features.


In [4]:
model_id = f"{PROJECT_ID}.superstore_data.flight_diverted_classifier"

create_model_sql = f"""
CREATE OR REPLACE MODEL `{model_id}`
OPTIONS(
  model_type='logistic_reg',
  input_label_cols=['is_diverted'],
  enable_global_explain=TRUE
) AS
SELECT
  CASE WHEN arrival_delay > 60 THEN 1 ELSE 0 END AS is_diverted, # Create a binary label
  departure_delay
FROM `{FULL_TABLE}`
WHERE arrival_delay IS NOT NULL AND departure_delay IS NOT NULL
LIMIT 500000;
"""
job = client.query(create_model_sql); job.result()
print("Model created:", model_id)

Model created: big-data-analysis-472319.superstore_data.flight_diverted_classifier


In [ ]:
# Get the table schema
table = client.get_table(FULL_TABLE)

# Print column names
print("Column names in the table:")
for field in table.schema:
    print(field.name)

Column names in the table:
date
airline
airline_code
departure_airport
departure_state
departure_lat
departure_lon
arrival_airport
arrival_state
arrival_lat
arrival_lon
departure_schedule
departure_actual
departure_delay
arrival_schedule
arrival_actual
arrival_delay



---
## Evaluate with `ML.EVALUATE` — Validate

Get **precision**, **recall**, **log_loss**, and other metrics. Also compute a **confusion matrix**.


In [5]:

eval_sql = f"SELECT * FROM ML.EVALUATE(MODEL `{model_id}`)"
eval_df = client.query(eval_sql).result().to_dataframe()
eval_df


,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,0.899796,0.810313,0.984611,0.852713,0.051433,0.9784


In [6]:
# Confusion matrix at default threshold
cm_sql = f"""
SELECT *
FROM ML.CONFUSION_MATRIX(MODEL `{model_id}`,
  (SELECT CASE WHEN arrival_delay > 60 THEN 1 ELSE 0 END AS is_diverted, departure_delay
   FROM `{FULL_TABLE}` WHERE arrival_delay IS NOT NULL AND departure_delay IS NOT NULL LIMIT 200000))
"""
cm_df = client.query(cm_sql).result().to_dataframe()
cm_df

,expected_label,_0,_1
0,0,186996,901
1,1,2542,9561



**Gemini Explainer Prompt:**

```python
'''prompt = Task:
Generate a complete evaluation query to validate a classification model.
Prompt:
Generate a full ML.EVALUATE SQL example in BigQuery ML to validate a classification model’s performance.
The query should:
Return key metrics such as precision, recall, accuracy, log_loss, f1_score, and roc_auc.
Include a confusion matrix using the ML.CONFUSION_MATRIX function.
Clearly show how the evaluation dataset is specified (e.g., using the EVALUATE clause with a validation table).
Include short, clear SQL comments explaining each step of the query.
```
Paste your explanation below.

The model is highly accurate and discriminative, with low error rates and strong probability calibration.
If the application prioritizes catching all positives (high recall), you might consider adjusting the prediction threshold slightly lower to reduce false negatives, but at the cost of a few more false positives.



---
## Threshold Tuning

By default, `ML.PREDICT` uses a threshold of **0.5**. You can change it to 0.75 and observe impacts on FP/FN.

> **Task:** Author your own Gemini prompt asking for an `ML.PREDICT` example that uses **`STRUCT(0.75 AS threshold)`** and explains when/why an airline might pick a higher threshold.


In [7]:
# Example scaffold: predictions with a higher threshold (0.75)
pred_sql = f"""
SELECT *
FROM ML.PREDICT(
  MODEL `{model_id}`,
  (SELECT
     30.0   AS departure_delay,  -- This is not used as a feature in the new model
     1400.0 AS departure_schedule,
     1600.0 AS arrival_schedule,
     'CA' AS arrival_state,
     'NY' AS departure_state,
     'AA' AS airline
  ),
  STRUCT(0.75 AS threshold)
)
"""
pred_df = client.query(pred_sql).result().to_dataframe()
pred_df

,predicted_is_diverted,predicted_is_diverted_probs,departure_delay,departure_schedule,arrival_schedule,arrival_state,departure_state,airline
0,0,"[{'label': 1, 'prob': 0.05005629352140551}, {'...",30.0,1400.0,1600.0,CA,NY,AA



---
## ✅ Deliverable for Lab 5

- Completed `Lab5_Classification_BQML.ipynb` with:
  - Business context write-up (FP vs FN)
  - `CREATE MODEL` SQL
  - `ML.EVALUATE` + `ML.CONFUSION_MATRIX` outputs and explanations
  - Threshold tuning example (`STRUCT(0.75 AS threshold)`)
- Push to **GitHub** and submit the link on **Brightspace**.
